# Cross-League Disciplinary Analysis
## Notebook 06 — Yellow Cards Forced: the Other Side of the Signal

Notebooks 04 and 05 established a consistent pattern: certain clubs, like Bayern Munich, Liverpool, Inter Milan, Juventus, Real Sociedad, commit fouls at a rate that generates fewer yellow cards than the rest of their league, and this signal persists across seasons when results are pooled.

There might be numbers of reason explaining *why*. Usually, football fans are very suspicious when they see data that, in some sense, seem to give an advantage to (their opposite) teams. Before analysing more data that could explain why a certain pattern emerge, for them already what we saw is the proof of a favorable attitude referees show to some teams. Of course, it doesn't apply to the team you are fan of. An easy way to straighten this feeling (we are sure that if fans don't have the result they want to they do not drop the suspicion) is to ash the following question:

> **If a club receives preferential treatment from referees, that advantage
> might appear not only when they commit fouls, but also when opponents foul
> them.**

In this notebook we apply the same bootstrap + Mann-Whitney pipeline to **fouls received** and **yellow cards forced** across all four leagues. If the same subset of clubs that showed a low-rate signal on the committed side also shows a high-rate signal on the forced side (while null clubs like Man City and Milan remain null) the case for a systematic pattern strengthens considerably.

## 1 The Data

In [1]:
# importing libraries and custom functions

import os, pickle, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, '../src')
from discipline_pipeline import run_boot_mw, z_screen, classify
from multi_season import fmt_season, fmt_pvalue


In [2]:
# importing data

# Load match-level data
with open('../data/processed/team_matches.pkl', 'rb') as f:
    tm = pickle.load(f)

In [3]:
# Compute yellow_cards_forced via self-merge:
# For each (season, date, team, opponent) row, yellow_cards_forced =
# the opponent's yellow_cards in the same match.
tm = tm.merge(
    tm[['season', 'date', 'team', 'opponent', 'yellow_cards']].rename(
        columns={'team': 'opponent', 'opponent': 'team',
                 'yellow_cards': 'yellow_cards_forced'}),
    on=['season', 'date', 'team', 'opponent'],
    how='left',
)

LEAGUES = {
    'Premier_League': 'Premier League',
    'La_Liga':        'La Liga',
    'Bundesliga':     'Bundesliga',
    'Serie_A':        'Serie A',
}

seasons          = sorted(tm['season'].unique())
readable_seasons = [fmt_season(s) for s in seasons]

print(f"Rows: {len(tm):,}  |  Leagues: {tm['league'].nunique()}  |  Seasons: {len(seasons)}")
print(f"New column — yellow_cards_forced: {tm['yellow_cards_forced'].notna().sum():,} non-null values")
print(tm[['team', 'opponent', 'fouls_received', 'yellow_cards_forced']].head(4).to_string(index=False))

Rows: 53,386  |  Leagues: 5  |  Seasons: 15
New column — yellow_cards_forced: 53,386 non-null values
    team  opponent  fouls_received  yellow_cards_forced
   Genoa     Lecce            13.0                  3.0
Sassuolo    Napoli            17.0                  1.0
   Milan Cremonese            14.0                  4.0
    Roma   Bologna            13.0                  1.0


### Data note

`yellow_cards_forced` is the total yellow cards received by the opposing team in that match, not exclusively cards awarded for fouls on the focal team, since per-incident card reasons are not available in this dataset. This is a known limitation: a yellow card for simulation, dissent, or time-wasting by the opponent is included. The same limitation applies symmetrically to `yellow_cards` on the committed side, so the two analyses remain comparable.

## Section 2 — Screening: which teams force yellow cards at an unusual rate?

We run the z-test for every team in every season across all four leagues,
using fouls received as the denominator and yellow cards forced as the
numerator.  This exploratory step is unconstrained — we do not pre-select
candidates — to see whether the significant teams on this metric overlap
with those identified in notebooks 04 and 05.

In [ ]:
# Build per-league season DataFrames (same structure as notebooks 04/05)
dfs = {}
for key in LEAGUES:
    league_df = tm[tm['league'] == key].copy()
    dfs[key] = {s: league_df[league_df['season'] == s] for s in seasons}

# Run z_screen on every season × league combination
# n_col = fouls_received, k_col = yellow_cards_forced
all_screen = []
for key, label in LEAGUES.items():
    for s in seasons:
        df_s = dfs[key].get(s, pd.DataFrame())
        if len(df_s) < 5:
            continue
        sc = z_screen(df_s, n_col='fouls_received', k_col='yellow_cards_forced')
        sc['season'] = s
        sc['league'] = label
        all_screen.append(sc)

screen = pd.concat(all_screen, ignore_index=True)
print(f"Total team-season observations screened: {len(screen):,}")
print(f"Significant at p < 0.05: {screen['significant'].sum()} "
      f"({screen['significant'].mean()*100:.1f}% — expected ~5% under H0)")

In [ ]:
# Sig-counts table per league — how often is each team significant?
# pct_present = sig_total / seasons the team was in the top flight
for key, label in LEAGUES.items():
    sc = screen[screen['league'] == label].copy()
    sig = sc[sc['significant']]

    seasons_present = sc.groupby('team')['season'].count().rename('seasons_present')
    total = sig.groupby('team')['season'].count().rename('sig_total')
    high  = (sig[sig['direction'] == 'high']
             .groupby('team')['season'].count().rename('sig_high'))
    low   = (sig[sig['direction'] == 'low']
             .groupby('team')['season'].count().rename('sig_low'))

    counts = (pd.concat([seasons_present, total, high, low], axis=1)
              .fillna(0).astype(int)
              .sort_values('sig_total', ascending=False)
              .reset_index())
    counts['pct_present'] = (counts['sig_total'] / counts['seasons_present'] * 100).round(1)
    counts = counts[counts['sig_total'] > 0]

    print(f"\n{'='*55}")
    print(f"{label} — teams significant in ≥1 season (forced cards)")
    print(f"  pct_present = sig_total / seasons_present (not % of table)")
    display(counts)

In [ ]:
# Stacked bar chart: significant teams per season, split by direction
# One panel per league
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)
axes = axes.flatten()

for ax, (key, label) in zip(axes, LEAGUES.items()):
    sc = screen[screen['league'] == label]
    sig = sc[sc['significant']]

    high_counts = (sig[sig['direction'] == 'high']
                   .groupby('season').size().reindex(seasons, fill_value=0))
    low_counts  = (sig[sig['direction'] == 'low']
                   .groupby('season').size().reindex(seasons, fill_value=0))

    x = range(len(seasons))
    ax.bar(x, high_counts, label='High (more cards forced)', color='#d6604d')
    ax.bar(x, low_counts, bottom=high_counts, label='Low (fewer cards forced)',
           color='#2166ac')
    ax.axhline(len(sc['team'].unique()) * 0.05, color='black',
               linewidth=1, linestyle=':', label='Expected 5%')
    ax.set_xticks(list(x))
    ax.set_xticklabels(readable_seasons, rotation=45, ha='right', fontsize=7)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_ylabel('Teams significant')

axes[0].legend(fontsize=8, frameon=False)
fig.suptitle('Significant teams per season — fouls received → cards forced',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 3 — Pooled pipeline: candidates across all four leagues

We pool all available seasons per candidate and run the full bootstrap +
Mann-Whitney pipeline on the fouls-received / yellow-cards-forced metric.

The candidate list mirrors notebooks 04 and 05 exactly — the same teams
tested for the fouls-committed signal — so results are directly comparable.

In [ ]:
# Candidates — identical to notebooks 04 and 05
CANDIDATES = {
    'Premier_League': ['Chelsea', 'Liverpool', 'Southampton',
                       'Arsenal', 'Man City', 'Man United'],
    'La_Liga':        ['Sociedad', 'Villarreal',
                       'Real Madrid', 'Barcelona', 'Ath Madrid'],
    'Bundesliga':     ['Augsburg', 'Ein Frankfurt', "M'gladbach",
                       'Bayern Munich', 'Dortmund'],
    'Serie_A':        ['Lazio', 'Napoli', 'Roma',
                       'Atalanta', 'Inter', 'Milan', 'Juventus'],
}

os.makedirs('../data/processed/notebook06', exist_ok=True)

# --- Load cached results if available ---
_cache_path = '../data/processed/notebook06/pooled_forced.pkl'
if os.path.exists(_cache_path):
    with open(_cache_path, 'rb') as f:
        pooled_forced = pickle.load(f)
    print("Loaded pooled forced-cards results from cache.")
else:
    pooled_forced = {key: {} for key in CANDIDATES}

    for key, label in LEAGUES.items():
        df_league = tm[tm['league'] == key].copy()
        teams = CANDIDATES[key]
        print(f"\nRunning {label} ({len(teams)} candidates)...")
        for team in teams:
            r = run_boot_mw(team, df_league,
                            n_col='fouls_received', k_col='yellow_cards_forced',
                            n_bootstrap=10_000, seed=42)
            pooled_forced[key][team] = r
            direction = 'high' if r['p_hat'] > r['p0'] else 'low'
            status = classify(r['p_boot'], r['p_mw'], direction)
            print(f"  {team:<20} r_rb={r['r_rb']:+.3f}  p_boot={r['p_boot']:.3f}"
                  f"  p_mw={r['p_mw']:.3f}  [{status}]")

    with open(_cache_path, 'wb') as f:
        pickle.dump(pooled_forced, f)
    print("\nResults cached.")

In [ ]:
# Summary tables — fouls received → cards forced
for key, label in LEAGUES.items():
    res = pooled_forced[key]
    rows = []
    for team, r in res.items():
        if r is None:
            continue
        rows.append(dict(
            team        = team,
            n           = r['n'],
            k           = r['k'],
            rate        = f"{r['p_hat']:.4f}",
            fpc_team    = f"{1/r['p_hat']:.1f}",
            league_rate = f"{r['p0']:.4f}",
            fpc_league  = f"{1/r['p0']:.1f}",
            p_boot      = fmt_pvalue(r['p_boot']),
            p_mw        = fmt_pvalue(r['p_mw']),
            r_rb        = f"{r['r_rb']:+.3f}",
        ))
    summary = pd.DataFrame(rows).sort_values('r_rb', ascending=False)
    print(f"\n{'='*60}")
    print(f"{label}  —  Pooled (fouls received → cards forced, sorted by r_rb)")
    print("  rate = yellow_cards_forced / fouls_received")
    print("  fpc  = fouls received per card forced (higher = harder to draw a card)")
    display(summary)

## Section 4 — Cross-analysis: fouls committed vs fouls received

We now place both results side by side for every candidate.  The key question
is whether the teams that showed a low-rate signal on the committed side also
show a high-rate signal on the forced side — and whether clubs that were null
on the committed side remain null here.

In [ ]:
# Load fouls-committed pooled results from notebooks 04 and 05 caches
committed_cache = {
    'Premier_League': '../data/processed/notebook05/pl_pooled_results.pkl',
    'La_Liga':        '../data/processed/notebook05/laliga_pooled_results.pkl',
    'Bundesliga':     '../data/processed/notebook05/bundesliga_pooled_results.pkl',
    'Serie_A':        '../data/processed/notebook04/pooled_results.pkl',
}
pooled_committed = {}
for key, path in committed_cache.items():
    with open(path, 'rb') as f:
        pooled_committed[key] = pickle.load(f)

# Build unified comparison DataFrame
rows = []
for key, label in LEAGUES.items():
    for team in CANDIDATES[key]:
        # Forced side (new)
        rf = pooled_forced[key].get(team)
        if rf is None:
            continue
        dir_f   = 'high' if rf['p_hat'] > rf['p0'] else 'low'
        stat_f  = classify(rf['p_boot'], rf['p_mw'], dir_f)

        # Committed side (notebooks 04/05)
        rc = pooled_committed[key].get(team, {})
        if isinstance(rc, dict) and 'p_hat' in rc:
            p_hat_c = float(rc['p_hat']); p0_c = float(rc['p0'])
            p_boot_c = float(rc['p_boot']); p_mw_c = float(rc['p_mw'])
            r_rb_c  = float(rc['r_rb'])
        else:
            p_hat_c = p0_c = p_boot_c = p_mw_c = r_rb_c = float('nan')
        dir_c  = 'low' if p_hat_c < p0_c else 'high'
        stat_c = classify(p_boot_c, p_mw_c, dir_c) if not np.isnan(p_boot_c) else 'N/A'

        rows.append(dict(
            league   = label,
            team     = team,
            r_rb_com = round(r_rb_c, 3),
            stat_com = stat_c,
            r_rb_for = rf['r_rb'],
            stat_for = stat_f,
        ))

comparison = pd.DataFrame(rows)
# Sort: committed r_rb ascending (most-low first)
comparison = comparison.sort_values(['league', 'r_rb_com'])

for label in LEAGUES.values():
    sub = comparison[comparison['league'] == label]
    print(f"\n{'='*70}")
    print(f"{label}")
    print(sub[['team','r_rb_com','stat_com','r_rb_for','stat_for']].to_string(index=False))

In [ ]:
# Cross-analysis scatter plot: r_rb committed (x) vs r_rb forced (y)
# One panel per league; colour by combined status
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

COLOR_MAP = {
    'Significant (low)':   '#2166ac',
    'Borderline (low)':    '#92c5de',
    'Null':                '#bbbbbb',
    'Borderline (high)':   '#f4a582',
    'Significant (high)':  '#d6604d',
    'N/A':                 '#eeeeee',
}

for ax, (key, label) in zip(axes, LEAGUES.items()):
    sub = comparison[comparison['league'] == label]

    for _, row in sub.iterrows():
        color = COLOR_MAP.get(row['stat_com'], '#bbbbbb')
        marker = 'D' if 'Significant' in str(row['stat_for']) else (
                 's' if 'Borderline' in str(row['stat_for']) else 'o')
        ax.scatter(row['r_rb_com'], row['r_rb_for'],
                   color=color, s=90, marker=marker,
                   edgecolors='black', linewidths=0.5, zorder=3)
        ax.annotate(row['team'], (row['r_rb_com'], row['r_rb_for']),
                    fontsize=7.5, textcoords='offset points', xytext=(5, 3))

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('$r_{rb}$ — fouls committed side', fontsize=10)
    ax.set_ylabel('$r_{rb}$ — fouls received side', fontsize=10)
    ax.set_title(label, fontsize=12, fontweight='bold')

# Legend for committed-side status (colour)
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_colors = [Patch(facecolor=c, edgecolor='black', label=l)
                 for l, c in COLOR_MAP.items() if l != 'N/A']
legend_shapes = [
    Line2D([0],[0], marker='D', color='w', markerfacecolor='grey',
           markersize=8, label='Significant (forced side)'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='grey',
           markersize=8, label='Borderline (forced side)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey',
           markersize=8, label='Null (forced side)'),
]
fig.legend(handles=legend_colors + legend_shapes,
           loc='lower center', ncol=4, fontsize=8,
           frameon=False, bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'Committed side (x) vs forced side (y)\n'
    'Colour = committed-side status   Shape = forced-side status',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Section 4 observations

*To be completed after reviewing the results above.*